# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My Baseline Rule

The rule is intentionally simple and transparent.

Content with higher Google Search Console clicks and impressions, together with better search position, receives a higher score.

Reason Codes

- HIGH_PERFORMER: High clicks and good search position.
- GROWING_CONTENT: Moderate performance with growth potential.
- LOW_VISIBILITY: Low clicks and poor search position.
- CTR_OPPORTUNITY: High impressions but relatively low clicks.

Action Labels

- Protect
- Monitor
- Improve SEO
- Rewrite

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
%pip -q install duckdb huggingface_hub

In [4]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

In [5]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [6]:
print("REL exists:", "REL" in globals())
print("con exists:", "con" in globals())

REL exists: True
con exists: True


In [7]:
query = f"""
SELECT
    CASE
        WHEN gsc_clicks = 0 THEN 'No Click'
        WHEN gsc_clicks BETWEEN 1 AND 10 THEN 'Low'
        WHEN gsc_clicks BETWEEN 11 AND 50 THEN 'Medium'
        ELSE 'High'
    END AS click_bucket,

    COUNT(*) AS n,
    ROUND(AVG(gsc_impressions),2) AS avg_impressions,
    ROUND(AVG(gsc_sum_position),2) AS avg_position

FROM {REL}

WHERE
    gsc_data_available IS TRUE

GROUP BY click_bucket

ORDER BY
CASE click_bucket
    WHEN 'No Click' THEN 1
    WHEN 'Low' THEN 2
    WHEN 'Medium' THEN 3
    WHEN 'High' THEN 4
END
"""

signal1 = con.sql(query).df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,click_bucket,n,avg_impressions,avg_position
0,No Click,3193080,46.14,639.87
1,Low,412711,293.60,2762.89
2,Medium,5087,2077.79,11167.37
3,High,183,8728.06,33797.90


In [8]:
con.sql(f"""
SELECT *
FROM {REL}
LIMIT 1
""").df().columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [9]:
query = f"""
SELECT
CASE
    WHEN gsc_clicks = 0 THEN 'No Click'
    WHEN gsc_clicks BETWEEN 1 AND 10 THEN 'Low'
    WHEN gsc_clicks BETWEEN 11 AND 50 THEN 'Medium'
    ELSE 'High'
END AS click_bucket,

COUNT(*) AS n,

ROUND(AVG(gsc_impressions),2) AS avg_impressions,

ROUND(AVG(gsc_avg_position),2) AS avg_position

FROM {REL}

WHERE
gsc_data_available IS TRUE

GROUP BY click_bucket

ORDER BY
CASE click_bucket
WHEN 'No Click' THEN 1
WHEN 'Low' THEN 2
WHEN 'Medium' THEN 3
WHEN 'High' THEN 4
END
"""

signal1 = con.sql(query).df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,click_bucket,n,avg_impressions,avg_position
0,No Click,3193080,46.14,16.75
1,Low,412711,293.60,8.79
2,Medium,5087,2077.79,5.20
3,High,183,8728.06,4.10


## Signal Check 1

**Signal:** Google Search Console Clicks

**Verdict:** **CONFIRMED**

The results show a clear relationship between clicks, impressions, and search position. Content with higher click counts also receives higher impressions and better average search positions. Therefore, Google Search Console clicks are a useful signal for identifying different content performance archetypes and can be safely used in the baseline rule.

In [10]:
query = f"""
SELECT
CASE
    WHEN gsc_avg_position <= 5 THEN 'Top 5'
    WHEN gsc_avg_position <= 10 THEN 'Top 10'
    WHEN gsc_avg_position <= 20 THEN 'Top 20'
    ELSE 'Beyond 20'
END AS position_bucket,

COUNT(*) AS n,

ROUND(AVG(gsc_clicks),2) AS avg_clicks,

ROUND(AVG(gsc_impressions),2) AS avg_impressions

FROM {REL}

WHERE gsc_data_available IS TRUE

GROUP BY position_bucket

ORDER BY
CASE position_bucket
WHEN 'Top 5' THEN 1
WHEN 'Top 10' THEN 2
WHEN 'Top 20' THEN 3
WHEN 'Beyond 20' THEN 4
END;
"""

signal2 = con.sql(query).df()

signal2

,position_bucket,n,avg_clicks,avg_impressions
0,Top 5,1263125,0.35,96.51
1,Top 10,920359,0.22,76.01
2,Top 20,519223,0.18,56.60
3,Beyond 20,908354,0.09,65.41


## Signal Check 2

**Signal:** Google Search Console Average Position

**Verdict:** **CONFIRMED**

The results indicate that better search positions are generally associated with higher clicks and stronger visibility. Content ranked in the Top 5 has the highest average clicks (0.35) and impressions (96.51), while clicks gradually decrease as the average position becomes worse (Top 10 → Top 20 → Beyond 20). Although the "Beyond 20" bucket shows slightly higher average impressions than the "Top 20" bucket, it still receives substantially fewer clicks, suggesting lower search effectiveness. Therefore, average search position is confirmed as a reliable signal for identifying different content performance archetypes and is suitable for the baseline scoring rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.